# SUC - Example 2 (Stochastic with 3 solar scenarios)

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)

Two-stage SUC with `Pg` indexed by scenario.

In [1]:
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint, SolverFactory,
    Binary, minimize, value
)

# ---- Data: loaded from external file 'RE_Intgrtn1_e2_SUC_data.txt' ----
import sys, pathlib
# Make the ampl_data parser importable (one level up from this notebook)
_pkg = pathlib.Path.cwd().parent
if str(_pkg) not in sys.path: sys.path.insert(0, str(_pkg))
from ampl_data import parse_ampl_data

_d = parse_ampl_data('RE_Intgrtn1_e2_SUC_data.txt')

GEN_data     = _d['GEN']
PERIOD_data  = _d['PERIOD']
SCEN_data    = _d['SCENARIO']
gen_min      = _d['gen_min']
gen_max      = _d['gen_max']
gen_OpCost   = _d['gen_OpCost']
gen_SuCost   = _d['gen_SuCost']
Time_TotalPd = _d['Time_TotalPd']
SolarProb    = _d['SolarProb']
SolarTotalP  = _d['SolarTotalP']

m = ConcreteModel()
m.GEN      = Set(initialize=GEN_data,    ordered=True)
m.PERIOD   = Set(initialize=PERIOD_data, ordered=True)
m.SCENARIO = Set(initialize=SCEN_data,   ordered=True)

m.gen_min    = Param(m.GEN, initialize=gen_min)
m.gen_max    = Param(m.GEN, initialize=gen_max)
m.gen_OpCost = Param(m.GEN, initialize=gen_OpCost)
m.gen_SuCost = Param(m.GEN, initialize=gen_SuCost)
m.Time_TotalPd = Param(m.PERIOD, initialize=Time_TotalPd)
m.SolarProb   = Param(m.SCENARIO, initialize=SolarProb)
m.SolarTotalP = Param(m.SCENARIO, initialize=SolarTotalP)

m.u  = Var(m.GEN, m.PERIOD, domain=Binary)
m.Pg = Var(m.GEN, m.PERIOD, m.SCENARIO)

m.obj = Objective(
    rule=lambda mm: sum(mm.SolarProb[s]*(mm.gen_OpCost[g]*mm.Pg[g,t,s] + mm.gen_SuCost[g]*mm.u[g,t])
                       for g in mm.GEN for t in mm.PERIOD for s in mm.SCENARIO),
    sense=minimize
)

m.PowerBalance = Constraint(m.PERIOD, m.SCENARIO,
    rule=lambda mm,t,s: sum(mm.Pg[g,t,s] for g in mm.GEN) == mm.Time_TotalPd[t] - mm.SolarTotalP[s])
m.genLimit_Min = Constraint(m.GEN, m.PERIOD, m.SCENARIO,
    rule=lambda mm,g,t,s: mm.gen_min[g]*mm.u[g,t] <= mm.Pg[g,t,s])
m.genLimit_Max = Constraint(m.GEN, m.PERIOD, m.SCENARIO,
    rule=lambda mm,g,t,s: mm.Pg[g,t,s] <= mm.gen_max[g]*mm.u[g,t])

model = m

In [2]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)
m = model
print("u (commitment):")
for g in m.GEN:
    for t in m.PERIOD:
        print(f"  u[{g},{t}] = {int(round(value(m.u[g,t])))}")
print("\nPg (per scenario):")
for g in m.GEN:
    for t in m.PERIOD:
        for s in m.SCENARIO:
            print(f"  Pg[{g},{t},{s}] = {value(m.Pg[g,t,s]):.4f}")

Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmpa8fgq344.pyomo.lp


Reading time = 0.00 seconds
x1: 15 rows, 8 columns, 30 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  90


MIPGap  0



Optimize a model with 15 rows, 8 columns and 30 nonzeros


Model fingerprint: 0x0946c895
Variable types: 6 continuous, 2 integer (2 binary)
Coefficient statistics:


  Matrix range     [1e+00, 9e+01]
  Objective range  [3e+00, 8e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [6e+01, 1e+02]


Presolve removed 15 rows and 8 columns
Presolve time: 0.00s


Presolve: All rows and columns removed



Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 20 available processors)

Solution count 1: 2100 

Optimal solution found (tolerance 0.00e+00)
Best objective 2.100000000000e+03, best bound 2.100000000000e+03, gap 0.0000%


ok optimal
u (commitment):
  u[1,1] = 1
  u[2,1] = 1

Pg (per scenario):
  Pg[1,1,1] = 40.0000
  Pg[1,1,2] = 60.0000
  Pg[1,1,3] = 80.0000
  Pg[2,1,1] = 20.0000
  Pg[2,1,2] = 20.0000
  Pg[2,1,3] = 20.0000
